# Lezione 52 — L'architettura del Memory AI Lab

> **Modulo:** Memory AI Lab completo (capstone) · **Tempo stimato:** 25 minuti
> **Prerequisiti:** aver seguito le Fasi 1-7 (Lezioni 1-51).
>
> Obiettivo pratico unico: fissare il **contratto di output** del progetto
> finale e lo **scheletro della pipeline** che, nelle Lezioni 53-60, riempiremo
> con i componenti gia' costruiti nel corso. Apre la Fase 8.

## Teoria essenziale

Il Memory AI Lab riceve una memoria testuale e produce un **record strutturato**.
Ogni campo e' il risultato di una fase del corso:

| campo | da dove viene |
|-------|---------------|
| `type` | classificatore (Fasi 2-3, Lezione 54) |
| `entities`, `relations` | estrazione/grafo (Lezioni 26-27, 55-56) |
| `importance`, `should_store` | scoring (Lezione 25) |
| `embedding` | embedding (Lezioni 16-17, 55) |

Fissare **ora** il contratto (uno schema esplicito, come la Lezione 22) permette
di costruire i componenti in modo indipendente e di testarne l'integrazione.

In [1]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class MemoryRecord:
    memory_id: str
    text: str
    timestamp: Optional[str] = None
    type: Optional[str] = None
    entities: list = field(default_factory=list)
    topics: list = field(default_factory=list)
    importance: Optional[float] = None
    should_store: Optional[bool] = None
    embedding_dim: Optional[int] = None          # teniamo la dimensione, non il vettore intero
    relations: list = field(default_factory=list)

def record_valido(r: MemoryRecord):
    errori = []
    if not r.memory_id:
        errori.append("memory_id mancante")
    if not r.text:
        errori.append("text mancante")
    if r.type is not None and r.type not in {"episodic", "semantic", "preference"}:
        errori.append(f"type non valido: {r.type}")
    if r.importance is not None and not (0.0 <= r.importance <= 1.0):
        errori.append("importance fuori [0,1]")
    return errori

esempio = MemoryRecord("mem_001", "Marco visited Glasgow with his son.")
print("record grezzo valido, errori:", record_valido(esempio))

record grezzo valido, errori: []


## Lo scheletro della pipeline

Definiamo i **componenti** come funzioni con una firma stabile (qui stub) e una
funzione `processa` che li orchestra. Nelle prossime lezioni sostituiremo gli
stub con le implementazioni reali, senza cambiare la struttura.

In [2]:
# stub: firme stabili, implementazioni reali nelle Lezioni 54-56
def classifica_tipo(testo):
    return "episodic"

def estrai_entita(testo):
    return []

def calcola_importanza(testo):
    return 0.5

def costruisci_relazioni(testo, entita):
    return []

def fai_embedding_dim(testo):
    return 32

def processa(memory_id, testo, timestamp=None, soglia_store=0.4):
    ent = estrai_entita(testo)
    imp = calcola_importanza(testo)
    r = MemoryRecord(
        memory_id=memory_id, text=testo, timestamp=timestamp,
        type=classifica_tipo(testo), entities=ent,
        importance=imp, should_store=imp >= soglia_store,
        embedding_dim=fai_embedding_dim(testo),
        relations=costruisci_relazioni(testo, ent),
    )
    return r

# smoke test: lo scheletro si assembla e produce un record valido
r = processa("mem_001", "Marco visited Glasgow with his son.")
assert record_valido(r) == []
assert r.should_store in (True, False)
print("pipeline (scheletro) OK ->", r.type, "| should_store:", r.should_store)

pipeline (scheletro) OK -> episodic | should_store: True


## Riepilogo (max 8 punti)

1. Il Memory AI Lab: memoria testuale → **record strutturato**.
2. Ogni campo del record e' il frutto di una fase del corso.
3. Fissare il **contratto** ora abilita sviluppo e test indipendenti.
4. Lo schema e' esplicito e validato (come la Lezione 22).
5. I componenti hanno **firme stabili** (qui stub).
6. `processa` li **orchestra** senza conoscerne l'implementazione.
7. Uno smoke test verifica l'assemblaggio end-to-end.
8. Le Lezioni 53-60 riempiono gli stub, struttura invariata.

## Quiz

1. Perche' definire il contratto di output prima dei componenti?
2. Cosa permette di fare avere firme stabili per i componenti?
3. Perche' teniamo `embedding_dim` e non il vettore intero nel record d'esempio?

*(Risposte: 1. per costruire e testare i pezzi in modo indipendente e integrarli
senza sorprese; 2. sostituire gli stub con le implementazioni reali senza
toccare l'orchestrazione; 3. per leggibilita' del record d'esempio — il vettore
completo vive nell'indice di similarita', non stampato inline.)*